# r256 validation — correctness first, speed second

Two commits are under test, and neither has ever run on a GPU:

| commit | change |
|---|---|
| `30a8ae3` | **fix**: r256's loop-carried staging row moved out of `%r25` |
| `e370965` | **perf**: register double-buffer the B fragment and scales |

### The bug being fixed

`gl_gemm_mma_q8_r256` stages 256 activation rows in **four passes per
k-block**, so it re-reads its staging row inside the k-loop:

```ptx
add.s32 %r42, %r25, 0;      // and +64, +128, +192
```

But the 32 m-tile bodies overwrite `%r25` on every single tile:

```ptx
ld.shared.u32 %r25, [%r35+16];   // second half of the A fragment
```

So k-block 0 stages correctly, its m-tiles destroy the row, and every k-block
after that stages from a row index that is really an A fragment. The parity
suite runs `in_dim=64`, i.e. **nb=2** — one good block, one garbage block —
which is the observed

```
gemm_mma_q8_r256(ntok=5)[0]: gpu 1.5951226 vs cpu 2.4754333
```

`gl_gemm_mma_q8` does not have this bug: it stages 64 rows in **one** pass and
consumes `%r25` into its pointers in the prologue, never reading it again.

The kernel already carried a note about this exact hazard for the staging
*byte offset*, which was moved to `%r45` for the same reason. Both
loop-carried values had the problem; one was fixed and one was missed.

### A falsifiable prediction

`[r256-ladder]` in `examples/bench.rs` sweeps shapes and reports the first
mismatch, with its own taxonomy: *"fails only when in_dim>32 => k-loop"*.

If the diagnosis is right, then **before** the fix the ladder breaks at
`in_dim=64` (nb=2) regardless of ntok, and **after** it is clean at every
shape. If the ladder still breaks, the diagnosis is wrong and nothing below it
matters.

### Order of gates, and why speed comes last

⛔ **A faster wrong answer is not a speedup.** r256's "41-43% faster"
benchmark measured a kernel that computes incorrect output. Parity gates the
performance numbers here: if correctness fails, this notebook reports no
timing at all.

⛔ **Expect zero change in prefill tok/s.** `runner.rs` calls `gemm_mma_q8`,
not r256 — grep says the only callers are the parity test and
`examples/bench.rs`. A correct, faster r256 changes end-to-end throughput by
exactly nothing until the engine is wired to it. That wiring is a separate
decision and is not made here.

## Step 1 — Config

In [ ]:
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BRANCH   = "glbench-vs-llamacpp"
GH_TOKEN = ""

# The commit that fixes the staging row. Used for the bisect fallback if
# parity fails at HEAD: testing it alone says which of the two commits broke.
FIX_ONLY_COMMIT = "30a8ae3"

import os, sys, re, json, time, shutil, subprocess

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.makedirs(WORK, exist_ok=True)
REPO_DIR = os.path.join(WORK, "gwenland-ai")
OUT_DIR  = WORK
PTX_REL  = os.path.join("glcuda", "src", "kernels", "glcuda_sm75.ptx")

def sh(cmd, cwd=None, timeout=7200, env=None):
    e = dict(os.environ)
    if env:
        e.update(env)
    try:
        p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True,
                           timeout=timeout, env=e, stdin=subprocess.DEVNULL)
        return p.returncode, p.stdout, p.stderr
    except subprocess.TimeoutExpired:
        return 124, "", f"timed out after {timeout}s"
    except Exception as ex:
        return 125, "", f"{type(ex).__name__}: {ex}"

rc, out, _ = sh(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                 "--format=csv,noheader"], timeout=60)
GPU_LINE = out.strip() if rc == 0 else "no nvidia-smi"
GPU_COUNT = len([l for l in GPU_LINE.splitlines() if l.strip()]) if rc == 0 else 0
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(f"gpus : {GPU_COUNT}")
for l in GPU_LINE.splitlines():
    print("      ", l)

RUN_META = {"date_utc": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
            "gpu": GPU_LINE.replace("\n", " | "), "gpu_count": GPU_COUNT}

if GPU_COUNT == 0:
    print("\n⛔ No GPU. The parity suite SKIPS without a device and reports "
          "nothing; every gate below would be vacuous.")


## Step 2 — Repo, and proof the code under test is present

The last A/B in this project ran eight times against a checkout that predated
the feature it was measuring. Both arms were identical and the result looked
plausible. So the source is checked for both changes before anything runs.

In [ ]:
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@") if GH_TOKEN else REPO_URL
    rc, o, e = sh(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR])
    print((o or e)[-1000:])
else:
    rc_f, _, e_f = sh(["git", "fetch", "--depth", "50", "origin", BRANCH], cwd=REPO_DIR)
    rc_r, _, e_r = sh(["git", "reset", "--hard", "FETCH_HEAD"], cwd=REPO_DIR)
    if rc_f != 0 or rc_r != 0:
        print("⛔ REFRESH FAILED - the clone is stale:")
        print("   fetch:", (e_f or "").strip()[:200])
        print("   reset:", (e_r or "").strip()[:200])
    else:
        print("refreshed existing clone")

rc, o, _ = sh(["git", "log", "--oneline", "-1"], cwd=REPO_DIR)
GL_COMMIT = o.strip()
print("commit :", GL_COMMIT)

ptx = ""
_p = os.path.join(REPO_DIR, PTX_REL)
if os.path.exists(_p):
    ptx = open(_p, encoding="utf-8", errors="replace").read()

r256 = ptx.split(".visible .entry gl_gemm_mma_q8_r256(")[-1] if ptx else ""
HAS_FIX  = "add.s32 %r42, %r46, 0;" in r256
HAS_PIPE = "%bfrag0n" in r256
STILL_BUGGY = "add.s32 %r42, %r25, 0;" in r256

print(f"staging fix (%r46) : {'present' if HAS_FIX else 'ABSENT'}")
print(f"double buffer      : {'present' if HAS_PIPE else 'ABSENT'}")
if STILL_BUGGY:
    print("⛔ the old `add.s32 %r42, %r25, 0` is still here - this checkout "
          "predates the fix.")

PRECONDITION_OK = HAS_FIX and HAS_PIPE and not STILL_BUGGY
if not PRECONDITION_OK:
    print("\n⛔ The code under test is not in this checkout. Every result "
          "below would describe something else. Update the clone first.")

if shutil.which("cargo") is None:
    os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | "
              "sh -s -- -y --default-toolchain stable --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
rc, o, _ = sh(["cargo", "--version"], timeout=180)
print("cargo  :", o.strip())


## Step 3 — Gate 0: does the PTX even JIT?

The PTX is hand-authored; `ptxas` compiles it at module load, and a malformed
instruction surfaces as a runtime `GlError::Engine`, never as a build failure.
Any `cargo test` that touches `KernelSet::load` exercises this.

In [ ]:
print("$ cargo test -p glcuda --lib\n")
rc_l, l_out, l_err = sh(["cargo", "test", "-p", "glcuda", "--lib"],
                        cwd=REPO_DIR, timeout=3600)
L_HAY = (l_out or "") + "\n" + (l_err or "")
for ln in L_HAY.splitlines():
    if ln.startswith("test result:") or "error" in ln.lower()[:40]:
        print(ln)

JIT_ERROR = None
for pat in [r"cuModuleLoadDataEx[^\n]*", r"CUDA_ERROR_INVALID_PTX[^\n]*",
            r"ptxas[^\n]*error[^\n]*"]:
    m = re.search(pat, L_HAY, re.I)
    if m:
        JIT_ERROR = m.group(0)
        break
print("\nPTX JIT:", "no error surfaced here" if not JIT_ERROR else f"⛔ {JIT_ERROR}")
print("(the lib tests do not all load the sm_75 module; Step 4 is the real "
      "JIT gate because parity calls the kernel)")


## Step 4 — Gate 1: parity on hardware

⛔ `glcuda/tests/parity.rs` **skips** without a CUDA device, printing
`SKIP: no CUDA driver/device on this machine`. A green summary can therefore
mean nothing ran — which is precisely the state r256 was blocked in. A skip is
reported here as proving nothing.

In [ ]:
def run_parity(tag):
    print(f"$ cargo test -p glcuda --test parity -- --test-threads=1   [{tag}]\n")
    rc, o, e = sh(["cargo", "test", "-p", "glcuda", "--test", "parity",
                   "--", "--test-threads=1"], cwd=REPO_DIR, timeout=7200)
    hay = (o or "") + "\n" + (e or "")
    skipped = "SKIP: no CUDA driver/device" in hay
    line = None
    for ln in hay.splitlines():
        s = ln.strip()
        if s.startswith("test ") and "r256_matches_dequantized_reference" in s and " ... " in s:
            line = s
    detail = None
    m = re.search(r"^gemm_mma_q8_r256\([^\n]*", hay, re.M)
    if m:
        detail = m.group(0).strip()
    if skipped:
        verdict = "SKIPPED - no CUDA device; proves nothing"
    elif line and line.endswith(" ok"):
        verdict = "PASSED on hardware"
    elif line and "FAILED" in line:
        verdict = "FAILED on hardware"
    elif line:
        verdict = f"ran, unclear: {line}"
    else:
        verdict = "test not found in output"
    return hay, verdict, line, detail

if not PRECONDITION_OK:
    P_HAY, R256_VERDICT, R256_LINE, R256_DETAIL = "", "NOT RUN - precondition failed", None, None
    print("skipped: the code under test is not present")
else:
    P_HAY, R256_VERDICT, R256_LINE, R256_DETAIL = run_parity("HEAD")
    for ln in P_HAY.splitlines():
        if ln.startswith("test result:") or "SKIP:" in ln or "r256" in ln:
            print(ln)

open(os.path.join(OUT_DIR, "r256_parity.txt"), "w", encoding="utf-8").write(P_HAY)
print(f"\ngemm_mma_q8_r256_matches_dequantized_reference: {R256_VERDICT}")
if R256_DETAIL:
    print("  ", R256_DETAIL)
PARITY_OK = R256_VERDICT == "PASSED on hardware"


### If parity failed: which of the two commits?

Two untested commits, one gate. If HEAD fails, the fix alone is tested by
restoring just the PTX from `30a8ae3` — that separates "the fix is wrong" from
"the double-buffer is wrong".

`git checkout <sha> -- <file>` rather than `git stash`: this repo has old
stashes parked, and a bare `pop` takes the wrong one.

In [ ]:
BISECT = None
if PRECONDITION_OK and R256_VERDICT == "FAILED on hardware":
    print("HEAD failed. Restoring the PTX from the fix-only commit...\n")
    rc, _, e = sh(["git", "checkout", FIX_ONLY_COMMIT, "--", PTX_REL], cwd=REPO_DIR)
    if rc != 0:
        print("could not check out the fix-only PTX:", (e or "")[:300])
    else:
        _, v2, _, d2 = run_parity(f"{FIX_ONLY_COMMIT} PTX only")
        BISECT = v2
        print(f"\nfix alone: {v2}")
        if d2:
            print("  ", d2)
        if v2 == "PASSED on hardware":
            print("\n=> the staging fix is correct; the double-buffer "
                  "introduced the failure.")
        elif v2 == "FAILED on hardware":
            print("\n=> the staging fix alone does not make parity pass. "
                  "Either the diagnosis is incomplete or there is a second "
                  "defect. The ladder in Step 5 localizes it by shape.")
    # Always restore HEAD's file so later steps measure what HEAD contains.
    sh(["git", "checkout", "HEAD", "--", PTX_REL], cwd=REPO_DIR)
    print("\nrestored HEAD's PTX")
else:
    print("not needed" if PARITY_OK else "skipped (parity did not fail on hardware)")


## Step 5 — Gate 2: the shape ladder

`[r256-ladder]` sweeps `(in_dim, ntok)` upward and reports the first mismatch
against the trusted `gemm_mma_q8`. Its own taxonomy: fails at `ntok=8` means
m-tile 0 or the epilogue; passes 8 but fails 16 means the m-tile advance;
**fails only when `in_dim>32` means the k-loop**; fails only above `ntok=64`
means the 4-pass staging.

The diagnosis predicts a clean ladder now. A break at `in_dim>32` would mean
the staging row was not the whole story.

In [ ]:
BENCH_LINES, B_HAY = [], ""
if not PRECONDITION_OK:
    print("skipped: precondition failed")
else:
    rc, o, e = sh(["cargo", "build", "--release", "-p", "glcuda", "--example", "bench"],
                  cwd=REPO_DIR, timeout=3600)
    if rc != 0:
        print((o + e)[-2500:])
        raise RuntimeError("bench example build failed")
    BENCH_BIN = os.path.join(REPO_DIR, "target", "release", "examples", "bench")
    rc, b_out, b_err = sh([BENCH_BIN], cwd=REPO_DIR, timeout=3600)
    B_HAY = (b_out or "") + "\n" + (b_err or "")
    open(os.path.join(OUT_DIR, "r256_bench.txt"), "w", encoding="utf-8").write(B_HAY)
    BENCH_LINES = [l.strip() for l in B_HAY.splitlines()
                   if re.search(r"\[(r256-ladder|r256-parity|gemm-phaseb|gemm-reuse|mma)", l)]
    for l in BENCH_LINES:
        print(l)
    if rc != 0:
        print(f"\n⛔ bench exited {rc} - a CUDA fault here is itself a result")

LADDER = [l for l in BENCH_LINES if "r256-ladder" in l]
MISMATCH = [l for l in LADDER if "MISMATCH" in l.upper()]
print()
print("ladder mismatches:", MISMATCH if MISMATCH else "none reported")


## Step 6 — Speed, only if correctness held

In [ ]:
PHASEB = [l for l in BENCH_LINES if "gemm-phaseb" in l]
REUSE  = [l for l in BENCH_LINES if "gemm-reuse" in l]

if not PARITY_OK:
    PERF_REPORTED = False
    print("⛔ NOT REPORTING TIMINGS.")
    print(f"   parity: {R256_VERDICT}")
    print("   A faster wrong answer is not a speedup. The 41-43% figure this")
    print("   kernel has carried for months was measured in exactly this")
    print("   state; repeating that would be repeating the mistake.")
else:
    PERF_REPORTED = True
    print("parity is green, so these numbers describe a kernel that computes")
    print("the right answer:\n")
    for l in PHASEB + REUSE:
        print("  ", l)
    print()
    print("⚠️ None of this changes prefill tok/s. runner.rs calls gemm_mma_q8;")
    print("   r256 is reached only from the parity test and this bench. Wiring")
    print("   it into the engine is a separate decision, and the earlier")
    print("   multi-stream A/B is a reminder that a kernel-level win does not")
    print("   have to survive the trip into production.")


## Step 7 — Write `R256_VALIDATION.md`

In [ ]:
NL = "\n"
B = []
def w(s=""):
    B.append(s)

w("# r256 validation")
w()
w(f"- **Date (UTC):** {RUN_META['date_utc']}")
w(f"- **GPU:** {RUN_META['gpu']}"
  + (f" - {RUN_META['gpu_count']} present, pinned to device 0" if RUN_META['gpu_count'] > 1 else ""))
w(f"- **Commit:** `{GL_COMMIT}`")
w(f"- **Staging fix present:** {HAS_FIX} · **Double buffer present:** {HAS_PIPE}")
w()
if not PRECONDITION_OK:
    w("⛔ **Nothing was measured.** The checkout does not contain the code "
      "under test, so every gate below would have described different code.")
    w()

w("## Gate 1: parity on hardware")
w()
w(f"**{R256_VERDICT}**")
if R256_LINE:
    w()
    w("```")
    w(R256_LINE)
    if R256_DETAIL:
        w(R256_DETAIL)
    w("```")
w()
if R256_VERDICT == "PASSED on hardware":
    w("The kernel now matches the dequantised CPU reference. The diagnosis "
      "holds: r256 stages 256 rows in four passes per k-block and therefore "
      "re-reads its staging row inside the k-loop, where the 32 m-tile bodies "
      "were overwriting it 32 times per block. `gl_gemm_mma_q8` is immune "
      "because it stages in one pass and never re-reads that register.")
    w()
    w("The `CUDA_ERROR_MISALIGNED_ADDRESS` that appeared when this kernel was "
      "wired into the engine is explained by the same defect: a garbage row "
      "index becomes a global address. That should be re-tested at the wire-in "
      "rather than assumed fixed.")
elif R256_VERDICT == "SKIPPED - no CUDA device; proves nothing":
    w("The suite skipped for want of a device. This is the same green summary "
      "the blocked state has always produced, and it says nothing about the "
      "kernel.")
elif R256_VERDICT == "FAILED on hardware":
    w("The kernel still does not match the reference.")
    if BISECT:
        w()
        w(f"Bisect - PTX from `{FIX_ONLY_COMMIT}` alone: **{BISECT}**")
        w()
        if BISECT == "PASSED on hardware":
            w("So the staging fix is correct and the double-buffer introduced "
              "the failure. Likely suspects, in order: the prefetch offsets "
              "(+32 / +48 / +2), the swap running for warps that never "
              "prefetched, or the last k-block swapping stale values in.")
        else:
            w("So the staging row was not the whole story. The ladder below "
              "localizes what remains by shape.")
w()

w("## Gate 2: shape ladder")
w()
if LADDER:
    w("```")
    for l in LADDER:
        w(l)
    w("```")
    w()
    w("First mismatch by shape, in the ladder's own taxonomy: `ntok=8` means "
      "m-tile 0 or the epilogue; passing 8 but failing 16 means the m-tile "
      "advance; failing only above `in_dim=32` means the k-loop; failing only "
      "above `ntok=64` means the 4-pass staging.")
    w()
    w(f"Mismatches reported: {len(MISMATCH)}")
else:
    w("No ladder output captured.")
w()

w("## Speed")
w()
if PERF_REPORTED:
    for l in PHASEB + REUSE:
        w(f"    {l}")
    w()
    w("⚠️ This does not move prefill throughput. `runner.rs` calls "
      "`gemm_mma_q8`; r256's only callers are the parity test and the bench. "
      "Wiring it in is a separate decision — and the multi-stream experiment "
      "is a standing reminder that a kernel-level win need not survive the "
      "trip into production.")
else:
    w("**Not reported.** Parity did not pass, and a faster wrong answer is not "
      "a speedup. The 41-43% figure this kernel has carried was measured in "
      "exactly that state.")
w()

w("## Appendix: raw output")
w()
for title, body in [("cargo test -p glcuda --test parity", P_HAY),
                    ("glcuda bench", B_HAY)]:
    w(f"### {title}")
    w()
    w("```")
    w((body or "(empty)").strip()[:16000])
    w("```")
    w()

MD_PATH = os.path.join(OUT_DIR, "R256_VALIDATION.md")
open(MD_PATH, "w", encoding="utf-8").write(NL.join(B))
print(f"wrote {MD_PATH} ({len(NL.join(B))} chars)")
print()
print("=" * 60)
print(f"parity  : {R256_VERDICT}")
print(f"ladder  : {len(MISMATCH)} mismatch(es)")
print(f"timings : {'reported' if PERF_REPORTED else 'withheld - correctness gate'}")
print("=" * 60)
